In [1]:
# Project Setup

from pathlib import Path
import os
import sys

PROJECT_ROOT = Path(os.getcwd()).resolve().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
# Imports

import importlib
import pandas as pd

import src.feature_engineering as fe

importlib.reload(fe)

from src.feature_engineering import (
    _validate_dataframe,
    _validate_columns_exist,
    get_numerical_columns,
)

In [3]:
# =============================================================================
# Load Dataset
# =============================================================================

from pathlib import Path

from src.preprocessing import load_dataset

DATA_PATH = Path("../data/interim/cleaned_data.csv").resolve()

df = load_dataset(DATA_PATH)

df.head()

,person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_grade,loan_amnt,loan_int_rate,loan_status,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length
0,22,59000,RENT,123.0,PERSONAL,D,35000,16.02,1,0.59,Y,3
1,21,9600,OWN,5.0,EDUCATION,B,1000,11.14,0,0.10,N,2
2,25,9600,MORTGAGE,1.0,MEDICAL,C,5500,12.87,1,0.57,N,3
3,23,65500,RENT,4.0,MEDICAL,C,35000,15.23,1,0.53,N,2
4,24,54400,RENT,8.0,MEDICAL,C,35000,14.27,1,0.55,Y,4


In [4]:
# =============================================================================
# Dataset Information
# =============================================================================

print(f"Rows    : {df.shape[0]}")
print(f"Columns : {df.shape[1]}")

df.info()

Rows    : 32416
Columns : 12
<class 'pandas.DataFrame'>
RangeIndex: 32416 entries, 0 to 32415
Data columns (total 12 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   person_age                  32416 non-null  int64  
 1   person_income               32416 non-null  int64  
 2   person_home_ownership       32416 non-null  str    
 3   person_emp_length           32416 non-null  float64
 4   loan_intent                 32416 non-null  str    
 5   loan_grade                  32416 non-null  str    
 6   loan_amnt                   32416 non-null  int64  
 7   loan_int_rate               32416 non-null  float64
 8   loan_status                 32416 non-null  int64  
 9   loan_percent_income         32416 non-null  float64
 10  cb_person_default_on_file   32416 non-null  str    
 11  cb_person_cred_hist_length  32416 non-null  int64  
dtypes: float64(3), int64(5), str(4)
memory usage: 3.0 MB


In [5]:
# =============================================================================
# Test : _validate_dataframe()
# =============================================================================

_validate_dataframe(df)

print("✅ _validate_dataframe() Passed")

✅ _validate_dataframe() Passed


In [6]:
# =============================================================================
# Test : _validate_columns_exist()
# =============================================================================

_validate_columns_exist(
    df,
    ["loan_status"]
)

print("✅ _validate_columns_exist() Passed")

✅ _validate_columns_exist() Passed


In [7]:
# =============================================================================
# Test : get_numerical_columns()
# =============================================================================

numerical_columns = get_numerical_columns(
    dataframe=df,
    target_column="loan_status"
)

print(f"Numerical Columns ({len(numerical_columns)}):")
print(numerical_columns)

Numerical Columns (7):
['person_age', 'person_income', 'person_emp_length', 'loan_amnt', 'loan_int_rate', 'loan_percent_income', 'cb_person_cred_hist_length']


## Numerical Feature Summary

The function successfully identified all numerical feature columns after excluding the target column. These columns will be used for numerical preprocessing in the upcoming pipeline.

In [8]:
from src.feature_engineering import get_categorical_columns

categorical_columns = get_categorical_columns(
    dataframe=df,
    target_column="loan_status"
)

print(categorical_columns)

['person_home_ownership', 'loan_intent', 'loan_grade', 'cb_person_default_on_file']


In [9]:
categorical_columns = get_categorical_columns(
    dataframe=df,
    target_column="loan_status",
    excluded_columns=["loan_grade"]
)

print(categorical_columns)

['person_home_ownership', 'loan_intent', 'cb_person_default_on_file']


In [10]:
get_categorical_columns(
    dataframe=df,
    target_column="default_status"
)

ValueError: The following required columns are missing from the DataFrame: ['default_status']

In [11]:
get_categorical_columns(
    dataframe=df,
    target_column="loan_status",
    excluded_columns=["loan_grade", 10]
)

TypeError: All values in 'excluded_columns' must be strings.

In [12]:
from src.feature_engineering import split_features_target
from src.config import TARGET_COLUMN

In [13]:
X, y = split_features_target(
    dataframe=df,
    target_column=TARGET_COLUMN
)

In [14]:
print("Feature Matrix Shape :", X.shape)
print("Target Vector Shape  :", y.shape)

Feature Matrix Shape : (32416, 11)
Target Vector Shape  : (32416,)


In [15]:
split_features_target(
    dataframe=df,
    target_column="default_status"
)

ValueError: The following required columns are missing from the DataFrame: ['default_status']

In [16]:
split_features_target(
    dataframe=df,
    target_column=10
)

TypeError: Expected 'target_column' to be a string.

# Train, Validation and Test Split

In [17]:
from src.feature_engineering import split_dataset

In [18]:
(
    X_train,
    X_validation,
    X_test,
    y_train,
    y_validation,
    y_test,
) = split_dataset(
    X,
    y,
)

In [19]:
print("Training Set")
print(X_train.shape)
print(y_train.shape)

print()

print("Validation Set")
print(X_validation.shape)
print(y_validation.shape)

print()

print("Test Set")
print(X_test.shape)
print(y_test.shape)

Training Set
(20745, 11)
(20745,)

Validation Set
(5187, 11)
(5187,)

Test Set
(6484, 11)
(6484,)


In [20]:
print("Original Distribution")
print(df["loan_status"].value_counts(normalize=True))

print()

print("Training Distribution")
print(y_train.value_counts(normalize=True))

print()

print("Validation Distribution")
print(y_validation.value_counts(normalize=True))

print()

print("Test Distribution")
print(y_test.value_counts(normalize=True))

Original Distribution
loan_status
0    0.781312
1    0.218688
Name: proportion, dtype: float64

Training Distribution
loan_status
0    0.781297
1    0.218703
Name: proportion, dtype: float64

Validation Distribution
loan_status
0    0.781377
1    0.218623
Name: proportion, dtype: float64

Test Distribution
loan_status
0    0.781308
1    0.218692
Name: proportion, dtype: float64


In [21]:
X_train.head()

,person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_grade,loan_amnt,loan_int_rate,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length
4846,23,30000,RENT,7.0,PERSONAL,C,5000,14.17,0.17,Y,3
20435,29,96500,RENT,13.0,PERSONAL,B,10000,8.88,0.10,N,8
22396,35,50000,RENT,6.0,VENTURE,B,7200,11.99,0.14,N,9
9428,22,62000,MORTGAGE,2.0,EDUCATION,A,10000,5.99,0.16,N,4
22032,32,52800,MORTGAGE,2.0,HOMEIMPROVEMENT,A,24000,7.51,0.45,N,10


In [22]:
y_train.head()

4846     1
20435    0
22396    0
9428     0
22032    0
Name: loan_status, dtype: int64

In [23]:
from src.feature_engineering import build_preprocessor

In [24]:
preprocessor = build_preprocessor(
    numerical_columns=numerical_columns,
    categorical_columns=categorical_columns,
)

In [25]:
preprocessor

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numerical', ...), ('categorical', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``

In [26]:
type(preprocessor)

sklearn.compose._column_transformer.ColumnTransformer

# Fit Preprocessing Pipeline

In [27]:
from src.feature_engineering import fit_preprocessor

In [28]:
preprocessor = fit_preprocessor(
    preprocessor=preprocessor,
    X_train=X_train,
)

In [31]:
preprocessor

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numerical', ...), ('categorical', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``

In [32]:
type(preprocessor)

sklearn.compose._column_transformer.ColumnTransformer

In [33]:
print(preprocessor)

ColumnTransformer(transformers=[('numerical',
                                 Pipeline(steps=[('scaler', StandardScaler())]),
                                 ['person_age', 'person_income',
                                  'person_emp_length', 'loan_amnt',
                                  'loan_int_rate', 'loan_percent_income',
                                  'cb_person_cred_hist_length']),
                                ('categorical',
                                 Pipeline(steps=[('encoder',
                                                  OneHotEncoder(handle_unknown='ignore'))]),
                                 ['person_home_ownership', 'loan_intent',
                                  'cb_person_default_on_file'])])


# Transform Dataset

In [34]:
from src.feature_engineering import transform_dataset

In [35]:
(
    X_train_processed,
    X_validation_processed,
    X_test_processed,
) = transform_dataset(
    preprocessor=preprocessor,
    X_train=X_train,
    X_validation=X_validation,
    X_test=X_test,
)

In [36]:
print("=" * 60)
print("Processed Dataset Shapes")
print("=" * 60)

print("Training :", X_train_processed.shape)
print("Validation :", X_validation_processed.shape)
print("Test :", X_test_processed.shape)

Processed Dataset Shapes
Training : (20745, 19)
Validation : (5187, 19)
Test : (6484, 19)


In [37]:
X_train_processed.head()

,numerical__person_age,numerical__person_income,numerical__person_emp_length,numerical__loan_amnt,numerical__loan_int_rate,numerical__loan_percent_income,numerical__cb_person_cred_hist_length,categorical__person_home_ownership_MORTGAGE,categorical__person_home_ownership_OTHER,categorical__person_home_ownership_OWN,categorical__person_home_ownership_RENT,categorical__loan_intent_DEBTCONSOLIDATION,categorical__loan_intent_EDUCATION,categorical__loan_intent_HOMEIMPROVEMENT,categorical__loan_intent_MEDICAL,categorical__loan_intent_PERSONAL,categorical__loan_intent_VENTURE,categorical__cb_person_default_on_file_N,categorical__cb_person_default_on_file_Y
4846,-0.744960,-0.542319,0.544813,-0.729132,1.024433,-0.002730,-0.691507,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0
20435,0.197992,0.458352,1.999275,0.065639,-0.693404,-0.658978,0.542647,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
22396,1.140944,-0.241366,0.302402,-0.379433,0.316516,-0.283979,0.789478,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0
9428,-0.902119,-0.060794,-0.667239,0.065639,-1.631882,-0.096479,-0.444676,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
22032,0.669468,-0.199232,-0.667239,2.290999,-1.138288,2.622264,1.036309,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0


In [38]:
print(X_train_processed.columns.tolist())

['numerical__person_age', 'numerical__person_income', 'numerical__person_emp_length', 'numerical__loan_amnt', 'numerical__loan_int_rate', 'numerical__loan_percent_income', 'numerical__cb_person_cred_hist_length', 'categorical__person_home_ownership_MORTGAGE', 'categorical__person_home_ownership_OTHER', 'categorical__person_home_ownership_OWN', 'categorical__person_home_ownership_RENT', 'categorical__loan_intent_DEBTCONSOLIDATION', 'categorical__loan_intent_EDUCATION', 'categorical__loan_intent_HOMEIMPROVEMENT', 'categorical__loan_intent_MEDICAL', 'categorical__loan_intent_PERSONAL', 'categorical__loan_intent_VENTURE', 'categorical__cb_person_default_on_file_N', 'categorical__cb_person_default_on_file_Y']


In [39]:
type(X_train_processed)

pandas.DataFrame

# Save and Load Preprocessor

In [40]:
from src.feature_engineering import (
    save_pickle,
    load_pickle,
)

from src.config import SCALER_PATH

In [41]:
save_pickle(
    object_to_save=preprocessor,
    file_path=SCALER_PATH,
)

In [42]:
loaded_preprocessor = load_pickle(
    SCALER_PATH,
)

In [43]:
type(loaded_preprocessor)

sklearn.compose._column_transformer.ColumnTransformer

In [44]:
print(loaded_preprocessor)

ColumnTransformer(transformers=[('numerical',
                                 Pipeline(steps=[('scaler', StandardScaler())]),
                                 ['person_age', 'person_income',
                                  'person_emp_length', 'loan_amnt',
                                  'loan_int_rate', 'loan_percent_income',
                                  'cb_person_cred_hist_length']),
                                ('categorical',
                                 Pipeline(steps=[('encoder',
                                                  OneHotEncoder(handle_unknown='ignore'))]),
                                 ['person_home_ownership', 'loan_intent',
                                  'cb_person_default_on_file'])])


# Final Verification

In [45]:
print("=" * 80)
print("TRAIN DATASET")
print("=" * 80)

X_train_processed.info()

print("\n")

print("=" * 80)
print("VALIDATION DATASET")
print("=" * 80)

X_validation_processed.info()

print("\n")

print("=" * 80)
print("TEST DATASET")
print("=" * 80)

X_test_processed.info()

TRAIN DATASET
<class 'pandas.DataFrame'>
Index: 20745 entries, 4846 to 16121
Data columns (total 19 columns):
 #   Column                                       Non-Null Count  Dtype  
---  ------                                       --------------  -----  
 0   numerical__person_age                        20745 non-null  float64
 1   numerical__person_income                     20745 non-null  float64
 2   numerical__person_emp_length                 20745 non-null  float64
 3   numerical__loan_amnt                         20745 non-null  float64
 4   numerical__loan_int_rate                     20745 non-null  float64
 5   numerical__loan_percent_income               20745 non-null  float64
 6   numerical__cb_person_cred_hist_length        20745 non-null  float64
 7   categorical__person_home_ownership_MORTGAGE  20745 non-null  float64
 8   categorical__person_home_ownership_OTHER     20745 non-null  float64
 9   categorical__person_home_ownership_OWN       20745 non-null  float64
 1

In [46]:
print("=" * 80)
print("Missing Values")
print("=" * 80)

print("Train      :", X_train_processed.isna().sum().sum())
print("Validation :", X_validation_processed.isna().sum().sum())
print("Test       :", X_test_processed.isna().sum().sum())

Missing Values
Train      : 0
Validation : 0
Test       : 0


In [47]:
print("=" * 80)
print("Processed Dataset Shapes")
print("=" * 80)

print("Training    :", X_train_processed.shape)
print("Validation  :", X_validation_processed.shape)
print("Test        :", X_test_processed.shape)

Processed Dataset Shapes
Training    : (20745, 19)
Validation  : (5187, 19)
Test        : (6484, 19)


In [48]:
print("=" * 80)
print("Feature Names")
print("=" * 80)

for column in X_train_processed.columns:
    print(column)

Feature Names
numerical__person_age
numerical__person_income
numerical__person_emp_length
numerical__loan_amnt
numerical__loan_int_rate
numerical__loan_percent_income
numerical__cb_person_cred_hist_length
categorical__person_home_ownership_MORTGAGE
categorical__person_home_ownership_OTHER
categorical__person_home_ownership_OWN
categorical__person_home_ownership_RENT
categorical__loan_intent_DEBTCONSOLIDATION
categorical__loan_intent_EDUCATION
categorical__loan_intent_HOMEIMPROVEMENT
categorical__loan_intent_MEDICAL
categorical__loan_intent_PERSONAL
categorical__loan_intent_VENTURE
categorical__cb_person_default_on_file_N
categorical__cb_person_default_on_file_Y


In [49]:
X_train_processed.describe().T

,count,mean,std,min,25%,50%,75%,max
numerical__person_age,20745.0,1.335800e-16,1.000024,-1.216436,-0.744960,-0.273484,0.355151,18.271241
numerical__person_income,20745.0,-1.164543e-17,1.000024,-0.933559,-0.414414,-0.166127,0.195017,89.292348
numerical__person_emp_length,20745.0,1.013838e-16,1.000024,-1.152060,-0.667239,-0.182418,0.544813,28.664420
numerical__loan_amnt,20745.0,-1.400877e-16,1.000024,-1.444426,-0.729132,-0.252269,0.431234,4.039496
numerical__loan_int_rate,20745.0,-3.699138e-17,1.000024,-1.816979,-0.820049,-0.008217,0.680216,3.963265
numerical__loan_percent_income,20745.0,2.226333e-17,1.000024,-1.596475,-0.752728,-0.190229,0.559769,5.716005
numerical__cb_person_cred_hist_length,20745.0,-3.767640e-17,1.000024,-0.938338,-0.691507,-0.444676,0.542647,5.972926
categorical__person_home_ownership_MORTGAGE,20745.0,4.149916e-01,0.492732,0.000000,0.000000,0.000000,1.000000,1.000000
categorical__person_home_ownership_OTHER,20745.0,3.470716e-03,0.058812,0.000000,0.000000,0.000000,0.000000,1.000000
categorical__person_home_ownership_OWN,20745.0,7.905519e-02,0.269831,0.000000,0.000000,0.000000,0.000000,1.000000


In [50]:
print("=" * 80)
print("Target Distribution")
print("=" * 80)

print(y_train.value_counts(normalize=True))

Target Distribution
loan_status
0    0.781297
1    0.218703
Name: proportion, dtype: float64
